In [1]:
import requests
import json
from fake_useragent import UserAgent
import time
import pandas as pd
import numpy as np
import math
import locale
from datetime import datetime, timedelta
import re
import os
from tqdm.auto import tqdm
from openai import OpenAI
import matplotlib.pyplot as plt
import smtplib
from email.message import EmailMessage
from email.header import Header
from email.utils import formataddr
from email import policy
import traceback
from io import BytesIO
from weasyprint import HTML
from zoneinfo import ZoneInfo

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 0. 相关方程

In [2]:
# 设置中国时区
CN_TZ = ZoneInfo("Asia/Shanghai")

def now_cn():
    return datetime.now(tz=CN_TZ)

In [3]:
def start_browser(user_data_dir=None, headless=False):
    opts = Options()
    opts.binary_location = CHROME_BINARY

    if headless:
        opts.add_argument("--headless=new")

    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    if user_data_dir:
        opts.add_argument(f"--user-data-dir={user_data_dir}")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

def cookies_to_header(driver):
    cookies = driver.get_cookies()
    return "; ".join(f"{c['name']}={c['value']}" for c in cookies)

def is_logged_in_weibo(driver) -> bool:
    """
    DOM/URL-based login detection (NOT cookie-based).
    Heuristics:
    - redirected to passport.weibo.com => not logged in
    - visible "登录" entry points => not logged in
    - visible user/home UI elements => logged in
    """
    url = (driver.current_url or "").lower()
    if "passport.weibo.com" in url:
        return False

    # Some common "not logged in" signals
    login_xpaths = [
        "//a[contains(@href,'passport.weibo.com') and (contains(.,'登录') or contains(.,'Log'))]",
        "//*[contains(text(),'登录') and (self::a or self::button)]",
        "//*[contains(text(),'扫码登录')]",
        "//*[contains(text(),'注册') and (self::a or self::button)]",
    ]
    for xp in login_xpaths:
        try:
            els = driver.find_elements(By.XPATH, xp)
            if any(e.is_displayed() for e in els):
                return False
        except Exception:
            pass

    # Some common "logged in" signals (header features vary; we use multiple)
    logged_in_xpaths = [
        "//*[contains(text(),'退出')]",                 # logout link sometimes exists
        "//*[contains(text(),'消息')]",                 # messages
        "//*[contains(text(),'发微博')]",               # post button
        "//a[contains(@href,'/u/')]",                   # profile link style
        "//a[contains(@href,'weibo.com/u/')]",          # profile link style
    ]
    for xp in logged_in_xpaths:
        try:
            els = driver.find_elements(By.XPATH, xp)
            if any(e.is_displayed() for e in els):
                return True
        except Exception:
            pass

    # Fallback: if neither found, treat as not logged in
    return False

def wait_for_weibo_login(driver, timeout_seconds=120, poll_seconds=2) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if is_logged_in_weibo(driver):
            return True
        time.sleep(poll_seconds)
    return False

def extract_status_id_from_textarea(driver, timeout=20) -> str:
    """
    Extract Weibo status ID from:
    <textarea id="comment-textarea-<STATUS_ID>">
    """

    # Wait until the comment textarea is present in the DOM
    textarea = WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located(
            (By.XPATH, "//textarea[starts-with(@id, 'comment-textarea-')]")
        )
    )

    textarea_id = textarea.get_attribute("id")
    # e.g. "comment-textarea-5263307886035659"

    match = re.match(r"comment-textarea-(\d+)", textarea_id)
    if not match:
        raise RuntimeError(f"Unexpected textarea id format: {textarea_id}")

    return match.group(1)


def open_weibo_and_get_cookie(post_url: str, user_data_dir="./chrome_profile_weibo", timeout_seconds=120):
    driver = start_browser(user_data_dir=user_data_dir, headless=False)
    driver.get(post_url)

    print("====================================================")
    print("Weibo login required.")
    print("A Chrome window has opened.")
    print("Please log in to Weibo (QR code / username).")
    print(f"You have {timeout_seconds} seconds (2 minutes).")
    print("After login, come back here — the script will continue automatically.")
    print("====================================================")

    ok = wait_for_weibo_login(driver, timeout_seconds=timeout_seconds, poll_seconds=2)
    if not ok:
        driver.quit()
        raise RuntimeError("Login not detected within 2 minutes. Please rerun and log in first.")

    cookie_header = cookies_to_header(driver)
    status_id = extract_status_id_from_textarea(driver)

    return driver, cookie_header, status_id

In [4]:
def _make_session(COOKIE_HEADER: str) -> requests.Session:
    """Build a configured requests.Session with Weibo cookies + headers."""
    s = requests.Session()
    s.headers.update({
        "User-Agent": UserAgent().random,
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://weibo.com/",
        "Cookie": COOKIE_HEADER,
        "Connection": "keep-alive",
    })
    # Warm up (refreshes session cookies)
    try:
        s.get("https://weibo.com/", timeout=15)
    except Exception:
        pass
    # Mirror XSRF token if present
    xsrf = s.cookies.get("XSRF-TOKEN")
    if xsrf:
        s.headers["X-XSRF-TOKEN"] = xsrf
    return s
 
 
def _fetch_replies_for_primary(
    session: requests.Session,
    BASE: str,
    primary_comment: dict,
    uid: str,
    per_page: int = 20,
    max_pages: int = 200,
    sleep_between: float = 0.6,
) -> list:
    """
    Fetch ALL replies (secondary comments) for a single primary comment.
 
    Uses the same buildComments endpoint but with:
      - fetch_level = 1    (ask for the reply thread)
      - id          = <primary comment id>   (NOT the status id)
      - uid         = post author uid
      - flow        = 0
      - max_id      = paginated cursor
 
    Stops when the API returns an empty page or max_id=0.
    """
    primary_id = primary_comment.get("id") or primary_comment.get("idstr")
    if not primary_id:
        return []
 
    # Skip API call if Weibo already told us there are zero replies.
    # total_number on a primary comment = reply count.
    total_expected = primary_comment.get("total_number")
    if total_expected == 0:
        return []
 
    params = {
        "flow": 0,
        "is_reload": 1,
        "id": str(primary_id),
        "is_show_bulletin": 2,
        "is_mix": 0,
        "fetch_level": 1,      # <-- the important switch
        "max_id": 0,
        "count": per_page,
        "uid": str(uid),
    }
 
    replies = []
    for page in range(1, max_pages + 1):
        try:
            r = session.get(BASE, params=params, timeout=15)
            r.raise_for_status()
            j = r.json()
        except Exception as e:
            print(f"  [WARN] reply fetch failed for primary {primary_id} page {page}: {e}")
            break
 
        if j.get("ok") == -100:
            raise RuntimeError("Login required (ok:-100): cookies expired / incomplete.")
 
        data = j.get("data", []) or []
        replies.extend(data)
 
        next_max_id = j.get("max_id", 0)
        if not data or not next_max_id:
            break
 
        params["max_id"] = next_max_id
        time.sleep(sleep_between)
 
    return replies
 
 
def fetch_all(COOKIE_HEADER, BASE, params, max_pages=500):
    """
    Fetch all primary comments, then for each primary comment with replies,
    fetch the FULL reply thread (the '共N条回复' secondary comments).
 
    The reply list is stored on each primary comment dict under the key
    '_all_replies' so that build_comments_df() can consume it.
 
    Args:
        COOKIE_HEADER: cookie string from the logged-in browser session
        BASE:          "https://weibo.com/ajax/statuses/buildComments"
        params:        dict with flow/id/uid/count/fetch_level=0/max_id=0 etc.
        max_pages:     max pagination pages for the PRIMARY comment stream
    """
    # ---- local copy so we never mutate the caller ----
    params_local = params.copy()
 
    s = _make_session(COOKIE_HEADER)
 
    # We need the post author's uid for reply-thread calls.
    # It's already in params (your notebook sets params["uid"] = uid).
    uid = params_local.get("uid")
 
    # ---- Stage 1: fetch all primary comments (your original logic) ----
    primaries = []
    for page in range(1, max_pages + 1):
        r = s.get(BASE, params=params_local, timeout=15)
        r.raise_for_status()
        j = r.json()
 
        if j.get("ok") == -100:
            raise RuntimeError("Login required (ok:-100): cookies expired / incomplete / mismatched.")
 
        data = j.get("data", []) or []
        primaries.extend(data)
 
        next_max_id = j.get("max_id", 0)
        print(f"[primary] page={page} got={len(data)} next_max_id={next_max_id}")
 
        if not data or not next_max_id:
            break
 
        params_local["max_id"] = next_max_id
        time.sleep(1)
 
    # ---- Stage 2: for each primary comment with replies, fetch the full thread ----
    total_with_replies = sum(
        1 for c in primaries if (c.get("total_number") or 0) > 0
    )
    print(f"[replies] {total_with_replies}/{len(primaries)} primary comments have replies; fetching...")
 
    fetched_so_far = 0
    for c in primaries:
        total_expected = c.get("total_number") or 0
 
        if total_expected == 0:
            c["_all_replies"] = []
            continue
 
        replies = _fetch_replies_for_primary(
            session=s,
            BASE=BASE,
            primary_comment=c,
            uid=uid,
        )
        c["_all_replies"] = replies
        fetched_so_far += 1
 
        # Light progress log every 20 threads
        if fetched_so_far % 20 == 0:
            print(f"[replies] fetched {fetched_so_far}/{total_with_replies} reply threads")
 
        # Gentle pacing between primary threads to be polite to the server
        time.sleep(0.4)
 
    print(f"[replies] done. Fetched {fetched_so_far} reply threads.")
    return primaries

In [5]:
def build_comments_df(comments):
    """
    comments: list of dicts returned by fetch_all (primary comments,
              each enriched with '_all_replies').
 
    Returns: DataFrame with columns:
      - posted_time
      - comment
      - comment_type (primary/secondary)
    """
    rows = []
 
    for c in comments:
        # 1) primary comment
        rows.append({
            "posted_time": c.get("created_at"),
            "comment": c.get("text_raw"),
            "comment_type": "primary",
        })
 
        # 2) secondary comments — prefer the FULL list we fetched
        children = c.get("_all_replies")
 
        # Fallback to the API preview list if for some reason _all_replies
        # wasn't populated (e.g., an unexpected error path).
        if not isinstance(children, list):
            children = c.get("comments") or []
 
        for cc in children:
            rows.append({
                "posted_time": cc.get("created_at"),
                "comment": cc.get("text_raw"),
                "comment_type": "secondary",
            })
 
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["posted_time", "comment", "comment_type"])
 
    # format
    df["posted_time"] = pd.to_datetime(
        df["posted_time"],
        format="%a %b %d %H:%M:%S %z %Y",
        errors="raise",
    ).dt.tz_localize(None)
 
    df["comment"] = df["comment"].astype(str)
    df["comment_type"] = df["comment_type"].astype(str)
 
    return df

In [6]:
# 情感分析方程
def deepseek_sentiment_3class(
    df: pd.DataFrame,
    rows_per_request: int,
    text_col: str = "comment",
    model: str = "deepseek-chat",
    max_retries: int = 5,
    sleep_base: float = 1.5,
) -> pd.DataFrame:
  texts = df[text_col].astype(str).fillna("").tolist()
  n = len(texts)

  labels = [None] * n
  confs  = [None] * n

  SYSTEM = """你是中文微博评论的情感分析器。
  你的任务是判断评论的整体情感倾向，只能输出以下三类之一：

  - POS：整体为正面（包括表扬、支持、送花、点赞、正向表情）
  - NEG：整体为负面（不满、批评、讽刺、抱怨）
  - MIXED：同时包含明显正面和负面，或态度矛盾

  规则：
  - 即使评论很短，只要语用上是支持或表扬，也可以判为 POS
  - 不要输出 NEU / UNKNOWN / 其他类别
  - 不要解释规则
  """

  for start in tqdm(range(0, n, rows_per_request), desc="DeepSeek sentiment (3-class)"):
      end = min(start + rows_per_request, n)
      batch = texts[start:end]

      payload = [{"i": start + j, "text": t} for j, t in enumerate(batch)]

      user_msg = (
          "请对以下评论逐条进行情感判断，只能输出严格 JSON。\n\n"
          "JSON 格式：\n"
          "{\n"
          '  "results": [\n'
          '    {"i": 0, "label": "POS|NEG|MIXED", "confidence": 0~1}, ...\n'
          "  ]\n"
          "}\n\n"
          f"评论列表：{json.dumps(payload, ensure_ascii=False)}"
      )

      for attempt in range(max_retries):
          try:
              resp = client.chat.completions.create(
                  model=model,
                  messages=[
                      {"role": "system", "content": SYSTEM},
                      {"role": "user", "content": user_msg},
                  ],
                  temperature=0,
                  response_format={"type": "json_object"},
              )

              data = json.loads(resp.choices[0].message.content)

              for r in data["results"]:
                  i = int(r["i"])
                  labels[i] = r["label"]
                  confs[i]  = float(r["confidence"])

              break

          except Exception:
              if attempt == max_retries - 1:
                  for i in range(start, end):
                      labels[i], confs[i] = "MIXED", 0.5
              else:
                  time.sleep(sleep_base * (2 ** attempt))

  return pd.DataFrame(
      {"sentiment": labels, "confidence": confs},
      index=df.index,
  )

In [7]:
#总结关键词方程
def summarize_keywords_with_examples(comments_df, model: str = "deepseek-chat"):
  comments = (
      comments_df["comment"]
      .astype(str)
      .fillna("")
      .tolist()
  )

  SYSTEM = """你是中文社交媒体评论分析助手。
  任务：从给定评论中提炼最重要的3个关键词，并为每个关键词挑选3条最能代表该关键词的原始评论（逐字引用）。
  要求：
  - 只输出 JSON（不要输出任何额外文字）
  - keywords 必须恰好 3 个
  - 每个 keyword 的 supporting_comments 必须恰好 3 条，且必须来自输入评论原文（逐字）
  - summary 用中文，不超过两句话
  - 关键词要尽量是“主题/实体/概念词”，避免过于泛化（如“感觉”“东西”）
  """

  user_msg = (
      "请从以下评论中提炼关键词并给出例句。\n"
      "输出严格 JSON，格式必须为：\n"
      "{\n"
      '  "keywords": [\n'
      '    {"keyword": "xxx", "supporting_comments": ["原始评论1","原始评论2","原始评论3"]},\n'
      '    {"keyword": "yyy", "supporting_comments": ["...","...","..."]},\n'
      '    {"keyword": "zzz", "supporting_comments": ["...","...","..."]}\n'
      "  ],\n"
      '  "summary": "两句话以内"\n'
      "}\n\n"
      "注意：supporting_comments 必须逐字来自输入评论，不要改写。\n\n"
      f"评论列表：{json.dumps(comments, ensure_ascii=False)}"
  )

  resp = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": SYSTEM},
          {"role": "user", "content": user_msg},
      ],
      temperature=0,
      response_format={"type": "json_object"},
  )

  return json.loads(resp.choices[0].message.content)

In [8]:
# ------------------------------------------------------------
# 2) Keying + incremental scoring (avoid re-calling DeepSeek)
# ------------------------------------------------------------
def make_comment_key(row: pd.Series) -> tuple:
    """
    A stable-ish key for detecting whether a comment is "new".
    If your API exposes a unique comment id, prefer that instead.
    """
    return (
        row.get("comment_type"),
        row.get("posted_time"),
        str(row.get("comment") or "").strip(),
        # int(row.get("like_count") or 0),
    )


def score_new_comments_only(new_df: pd.DataFrame, rows_per_request_cap: int = 50) -> pd.DataFrame:
    """
    Calls deepseek_sentiment_3class on ONLY new rows.
    Expects deepseek_sentiment_3class returns columns: sentiment, confidence.
    """
    if new_df.empty:
        out = new_df.copy()
        out["sentiment"] = pd.Series(dtype=str)
        out["confidence"] = pd.Series(dtype=float)
        return out

    rows_per_request = min(rows_per_request_cap, len(new_df))
    evaled = deepseek_sentiment_3class(
        new_df,
        text_col="comment",
        rows_per_request=rows_per_request,
    )
    return new_df.join(evaled)


def attach_sentiment_from_master(df_latest: pd.DataFrame, df_master: pd.DataFrame) -> pd.DataFrame:
    """
    Attach sentiment/confidence onto latest-visible comments using keys from the master history.
    """
    latest = df_latest.copy()
    latest["_k"] = latest.apply(make_comment_key, axis=1)

    if df_master.empty:
        latest["sentiment"] = np.nan
        latest["confidence"] = np.nan
        return latest.drop(columns=["_k"])

    master = df_master.copy()
    master["_k"] = master.apply(make_comment_key, axis=1)

    out = latest.merge(
        master[["_k", "sentiment", "confidence"]],
        on="_k",
        how="left",
    )
    return out.drop(columns=["_k"])


# ------------------------------------------------------------
# 3) Stats + Reporting (VISIBLE ONLY; no plots)
# ------------------------------------------------------------
def compute_basic_stats(df_scored: pd.DataFrame) -> dict:
    """
    Returns counts + percentages for POS/NEG/MIXED and also UNKNOWN (not yet scored).
    Percentages are over total visible rows.
    """
    if df_scored is None or df_scored.empty:
        return {
            "total": 0,
            "counts": {"POS": 0, "NEG": 0, "MIXED": 0, "UNKNOWN": 0},
            "percents": {"POS": 0.0, "NEG": 0.0, "MIXED": 0.0, "UNKNOWN": 0.0},
        }

    s = df_scored.get("sentiment", pd.Series([np.nan] * len(df_scored)))
    total = int(len(df_scored))

    pos = int((s == "POS").sum())
    neg = int((s == "NEG").sum())
    mixed = int((s == "MIXED").sum())
    unknown = int(s.isna().sum())

    def pct(x: int) -> float:
        return round(100.0 * x / total, 2) if total > 0 else 0.0

    return {
        "total": total,
        "counts": {"POS": pos, "NEG": neg, "MIXED": mixed, "UNKNOWN": unknown},
        "percents": {"POS": pct(pos), "NEG": pct(neg), "MIXED": pct(mixed), "UNKNOWN": pct(unknown)},
    }


def compute_duplicate_comment_summary(df_visible_scored: pd.DataFrame) -> pd.DataFrame:
    """
    Duplicate summary for visible comments only, grouped by exact comment text.
    Returns columns: comment, freq, sentiment
    """
    if df_visible_scored is None or df_visible_scored.empty:
        return pd.DataFrame(columns=["comment", "freq", "sentiment"])

    dup_rows = df_visible_scored[df_visible_scored.duplicated(subset="comment", keep=False)].copy()
    if dup_rows.empty:
        return pd.DataFrame(columns=["comment", "freq", "sentiment"])

    # Prefer a non-null sentiment if possible
    def first_non_null(x):
        x = x.dropna()
        return x.iloc[0] if len(x) else np.nan

    dup_comment_summary = (
        dup_rows
        .groupby("comment", as_index=False)
        .agg(
            freq=("comment", "size"),
            sentiment=("sentiment", first_non_null),
        )
        .sort_values(by="freq", ascending=False)
        .reset_index(drop=True)
    )

    return dup_comment_summary[dup_comment_summary['freq'] > 2]


def render_report_visible_only(
    latest_visible_scored: pd.DataFrame,
    report_title: str,
) -> dict:
    """
    Report uses ONLY currently visible comments (latest scrape, with sentiment attached).
    - summary stats: counts + percents
    - top-5 primary by popularity (API order) + sentiment
    - keyword summary over visible comments
    """
    ts = now_cn()
    stats = compute_basic_stats(latest_visible_scored)

    print("\n" + "=" * 90)
    print(f"[REPORT] {report_title}")
    print(f"Time: {ts.strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 90)
    c = stats["counts"]
    p = stats["percents"]
    print(
        f"Visible total={stats['total']} | "
        f"POS={c['POS']} ({p['POS']}%)  "
        f"NEG={c['NEG']} ({p['NEG']}%)  "
        f"MIXED={c['MIXED']} ({p['MIXED']}%)  "
        f"UNKNOWN={c['UNKNOWN']} ({p['UNKNOWN']}%)"
    )
    print(
        f"所有评论={stats['total']} | "
        f"好评={c['POS']} ({p['POS']}%)  "
        f"差评={c['NEG']} ({p['NEG']}%)  "
        f"中性={c['MIXED']} ({p['MIXED']}%)  "
        f"未评分={c['UNKNOWN']} ({p['UNKNOWN']}%)"
    )
    print("=" * 90)

    # Top-5 primary by popularity (API order)
    top5_primary = latest_visible_scored[latest_visible_scored["comment_type"] == "primary"].head(5).copy().drop(columns=['comment_type'])
    if not top5_primary.empty:
        print("\nTop-5 Comments by Popularity:")
        print("热评前5条：")
        display_cols = ["posted_time", "sentiment", "confidence", "comment"]
        display(top5_primary[display_cols])

    # Duplicate comment summary (visible only)
    dup_comment_summary = compute_duplicate_comment_summary(latest_visible_scored)

    if not dup_comment_summary.empty:
        print("\nDuplicate Comment Summary:")
        print("重复评论总结：")
        display(dup_comment_summary)
    else:
        print("\nDuplicate Comment Summary: none")
        print("重复评论总结：无")

    # Build "Negative & Mixed Comments" table (NOT POS)
    cols = ["posted_time", "comment", "sentiment", "confidence"]

    df_neg_mixed = (
        latest_visible_scored.loc[latest_visible_scored["sentiment"] != "POS", cols]
        .copy()
    )
    
    # optional: sort newest first if posted_time is sortable
    if "posted_time" in df_neg_mixed.columns:
        try:
            df_neg_mixed = df_neg_mixed.sort_values("posted_time", ascending=False)
        except Exception:
            pass

    # Keyword summary (visible only)
    keyword_result = None
    if latest_visible_scored is not None and not latest_visible_scored.empty:
        kw_df = latest_visible_scored.copy()

        if not kw_df.empty:
            keyword_result = summarize_keywords_with_examples(kw_df)

            if keyword_result:
                print("\n关键字总结:")
                for item in keyword_result.get("keywords", []):
                    print("关键词:", item.get("keyword"))
                    for cc in item.get("supporting_comments", []):
                        print("  -", cc)
                print("\n总结:", keyword_result.get("summary"))

    return {
        "time": ts,
        "title": report_title,
        "visible_stats": stats,
        "top5_primary": top5_primary,
        "dup_comment_summary": dup_comment_summary,
        "neg_mixed_comments": df_neg_mixed,
        "keywords": keyword_result,
        "post_url": post_url
    }


# ------------------------------------------------------------
# 4) Cadence + schedule-aligned polling
# ------------------------------------------------------------
def get_poll_interval_seconds(elapsed: timedelta) -> int:
    """
    Poll cadence:
      0-15m: every 3m
      15-60m: every 5m
      1-4h: every 15m
      4-96h: every 1h
    """
    m = elapsed.total_seconds() / 60.0
    if m < 15:
        return 3 * 60
    if m < 60:
        return 5 * 60
    if m < 240:
        return 15 * 60
    return 60 * 60


def compute_next_poll_time(start_time: datetime, now: datetime) -> datetime:
    """
    Returns the next scheduled poll boundary time, aligned to the cadence grid.
    If a run overruns, the next poll aligns to the next boundary (e.g., 0,3,6,... minutes).

    Phase anchors:
      - 0-15m  grid anchored at start_time
      - 15-60m grid anchored at start_time + 15m
      - 1-4h   grid anchored at start_time + 1h
      - 4-96h  grid anchored at start_time + 4h
    """
    elapsed = now - start_time
    interval_s = get_poll_interval_seconds(elapsed)

    if elapsed < timedelta(minutes=15):
        phase_start = start_time
    elif elapsed < timedelta(hours=1):
        phase_start = start_time + timedelta(minutes=15)
    elif elapsed < timedelta(hours=4):
        phase_start = start_time + timedelta(hours=1)
    else:
        phase_start = start_time + timedelta(hours=4)

    delta_s = (now - phase_start).total_seconds()
    k = int(delta_s // interval_s) + 1  # next boundary strictly after now
    return phase_start + timedelta(seconds=k * interval_s)


def build_report_schedule(start_time: datetime) -> list[datetime]:
    """
    Scheduled reports:
      +15m, +1h, +4h, then every 4h until 24h, and final at 96h
    """
    times = [
        start_time, # delete when done with testing
        start_time + timedelta(minutes=15),
        start_time + timedelta(hours=1),
        start_time + timedelta(hours=4),
        start_time + timedelta(hours=8),
        start_time + timedelta(hours=12),
        start_time + timedelta(hours=16),
        start_time + timedelta(hours=20),
        start_time + timedelta(hours=24),
        start_time + timedelta(hours=96),
    ]
    return sorted(times)

In [9]:
# 发邮件方程
def _df_to_html(df: pd.DataFrame, max_rows: int = 100) -> str:
    if df is None or (hasattr(df, "empty") and df.empty):
        return "<p><em>(none)</em></p>"
    d = df.head(max_rows).copy()
    # keep long comments readable in HTML
    for col in d.columns:
        if d[col].dtype == "object":
            d[col] = d[col].astype(str)
    html = d.to_html(index=False, escape=True, classes="dataframe", border=1)
    html = html.replace("<th>", '<th style="text-align:left;">')
    html = html.replace("<td>", '<td style="text-align:left;">')
    return html

def render_report_dict_to_html(report: dict) -> str:
    """
    Takes the output of render_report_visible_only(...) and returns an HTML email body.
    """
    ts = report.get("time")
    if isinstance(ts, datetime):
        ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
    else:
        ts_str = str(ts)

    title = report.get("title", "Weibo Sentiment Report")
    post_url = report.get('post_url')

    stats = (report.get("visible_stats") or {})
    total = stats.get("total", "")
    counts = (stats.get("counts") or {})
    percents = (stats.get("percents") or {})

    top5 = report.get("top5_primary")
    dup_sum = report.get("dup_comment_summary")
    neg_mixed = report.get("neg_mixed_comments")
    keywords = report.get("keywords") or {}
    kw_items = keywords.get("keywords", []) if isinstance(keywords, dict) else []
    kw_summary = keywords.get("summary", "") if isinstance(keywords, dict) else ""

    def en_cn(en, cn):
        return f"""
        <div>{en}</div>
        <div style="color:#555;">{cn}</div>
        """

    html = f"""
    <html>
      <body style="font-family: Arial, sans-serif; line-height: 1.4;">

        <style>
          /* Make pandas table column headers left-aligned */
          table.dataframe th {{
            text-align: left !important;
          }}

          /* (optional) also keep cells left-aligned for consistency */
          table.dataframe td {{
            text-align: left;
          }}

          /* (optional) keep gridlines looking consistent */
          table.dataframe {{
            border-collapse: collapse;
          }}
          table.dataframe th, table.dataframe td {{
            border: 1px solid #999;
            padding: 4px 6px;
            vertical-align: top;
          }}
        </style>
    """
    
    # ---- Title ----
    html += en_cn(f"<h2>{title}</h2>", f"<h2>微博评论监测报告</h2>")

    # ---- URL (Chinese only, BEFORE time) ----
    if post_url:
        html += f"""
        <div style="color:#555;">
          <b>原微博链接：</b>
          <a href="{post_url}" target="_blank" rel="noopener noreferrer">{post_url}</a>
        </div>
        """
    else:
        html += """
        <div style="color:#555;">
          <b>原微博链接：</b>（未提供）
        </div>
        """

    # ---- Time ----
    html += f"""
    <div style="color:#555;"><b>发出时间：</b> {ts_str}</div>
    """

    html += "<hr/>"

    # ---- Visible Stats ----
    html += en_cn("<h3>Current Summary Statistics</h3>", "<h3>当前可见评论统计</h3>")

    html += en_cn(
        f"<b>Visible total:</b> {total}",
        f"<b>当前可见评论总数：</b> {total}"
    )

    html += f"""
    <ul>
      <li>
        {en_cn(f"<b>POS</b>: {counts.get('POS',0)} ({percents.get('POS',0)}%)",
               f"<b>正面</b>: {counts.get('POS',0)} ({percents.get('POS',0)}%)")}
      </li>
      <li>
        {en_cn(f"<b>NEG</b>: {counts.get('NEG',0)} ({percents.get('NEG',0)}%)",
               f"<b>负面</b>: {counts.get('NEG',0)} ({percents.get('NEG',0)}%)")}
      </li>
      <li>
        {en_cn(f"<b>MIXED</b>: {counts.get('MIXED',0)} ({percents.get('MIXED',0)}%)",
               f"<b>中性</b>: {counts.get('MIXED',0)} ({percents.get('MIXED',0)}%)")}
      </li>
      <li>
        {en_cn(f"<b>UNKNOWN</b>: {counts.get('UNKNOWN',0)} ({percents.get('UNKNOWN',0)}%)",
               f"<b>未评</b>: {counts.get('UNKNOWN',0)} ({percents.get('UNKNOWN',0)}%)")}
      </li>
    </ul>
    """

    # ---- Top 5 ----
    html += "<hr/>"
    html += en_cn("<h3>Top-5 Comments by Popularity</h3>", "<h3>热评前5条</h3>")
    html += _df_to_html(top5, max_rows=5)

    # ---- Duplicate Summary ----
    html += "<hr/>"
    html += en_cn("<h3>Duplicate Comment Summary</h3>", "<h3>重复评论汇总</h3>")
    html += _df_to_html(dup_sum, max_rows=50)

    # ---- Negative & Mixed Comments ----
    html += "<hr/>"
    html += en_cn("<h3>Negative & Mixed Comments</h3>", "<h3>负面与中性评论</h3>")

    neg_mixed = report.get("neg_mixed_comments")
    html += _df_to_html(neg_mixed, max_rows=200)  # adjust max_rows as needed
    
    # ---- Keywords ----
    html += "<hr/>"
    html += en_cn("<h3>Keywords</h3>", "<h3>关键词分析</h3>")

    if kw_items:
        html += "<ol>"
        for item in kw_items:
            kw = item.get("keyword", "")
            scom = item.get("supporting_comments", []) or []
            html += f"<li><b>{kw}</b><ul>"
            for c in scom[:3]:
                html += f"<li>{str(c)}</li>"
            html += "</ul></li>"
        html += "</ol>"
    else:
        html += en_cn("(no keyword result)", "（暂无关键词总结）")

    if kw_summary:
        html += f"<p><b>总结:</b> {kw_summary}</p>"

    html += """
        <hr/>
        <div style="font-size:12px; color:#777;">
          Generated by weibo monitoring notebook.<br/>
          本报告由微博情绪监控系统自动生成。
        </div>
      </body>
    </html>
    """
    
    return html


def _inject_pdf_css(html: str, *, scale: float = 0.9) -> str:
    """
    Inject PDF CSS to:
      - make a single-page layout more likely
      - tighten margins/fonts
      - avoid page breaks inside tables/headers
      - add consistent table gridlines
    """
    pdf_css = f"""
    <style>
      /* Page setup */
      @page {{
        size: A4;     /* portrait by default */
        margin: 8mm 8mm 2mm 8mm;
      }}

      /* Overall typography */
      body {{
        font-family: Arial, sans-serif;
        font-size: 10px;
        line-height: 1.25;
        color: #111;
      }}

      /* "Shrink-to-fit" container (WeasyPrint supports transform) */
      .pdf-scale {{
        transform: scale({scale});
        transform-origin: top left;
        width: calc(100% / {scale});
      }}

      h1, h2, h3 {{
        margin: 6px 0 4px 0;
        page-break-after: avoid;
      }}
      hr {{
        margin: 6px 0;
      }}

      /* Avoid breaking sections/tables across pages (best effort) */
      table, thead, tbody, tr, td, th {{
        page-break-inside: avoid;
      }}

      /* Gridlines + table look */
      table.dataframe {{
        width: 100%;
        border-collapse: collapse;
        table-layout: fixed;
        font-size: 9px;
      }}
      table.dataframe th, table.dataframe td {{
        border: 1px solid #666;
        padding: 3px 4px;
        vertical-align: top;
        word-wrap: break-word;
        overflow-wrap: break-word;
        text-align: left;
      }}
      table.dataframe th {{
        background: #f2f2f2;
        font-weight: 600;
        text-align: left;
      }}
    </style>
    """

    html = (html or "").replace("\xa0", " ")

    # Insert CSS right after <head> if present, else create a head.
    if "<head>" in html:
      html = html.replace("<head>", "<head>" + pdf_css, 1)
    else:
      html = html.replace("<html>", "<html><head>" + pdf_css + "</head>", 1)

    # Wrap body content in scaling container
    if "<body" in html:
        html = re.sub(r"(<body[^>]*>)", r"\1<div class='pdf-scale'>", html, count=1)
        html = html.replace("</body>", "</div></body>", 1)
    return html


def html_to_pdf_bytes(html: str, *, scale: float = 0.9) -> bytes:
    html = _inject_pdf_css(html, scale=scale)
    try:
        return HTML(string=html).write_pdf()
    except Exception as e:
        print(f"[PDF] WeasyPrint not available or failed. Reason: {e}")

def build_dynamic_pdf_name(report: dict) -> str:
    """
    Extract milestone duration from report title and build filename like:
    weibo_report_15m.pdf
    weibo_report_1h.pdf
    weibo_report_4h.pdf
    """

    title = report.get("title", "").lower()

    # Try to extract patterns like 15m, 1h, 4h, 24h
    match = re.search(r"(\d+)\s*(m|h)", title)

    if match:
        value = match.group(1)
        unit = match.group(2)
        return f"weibo_report_{value}{unit}.pdf"

    # Fallback: timestamp if milestone not detected
    ts = report.get("time", now_cn())
    if isinstance(ts, datetime):
        ts_str = ts.strftime("%Y%m%d_%H%M%S")
    else:
        ts_str = "null_time"

    return f"weibo_report_{ts_str}.pdf"


def send_gmail_html(
    *,
    subject: str,
    html_body: str,
    to_email: str,
    from_email: str | None = None,
    app_password: str | None = None,
    attach_pdf: bool = True,
    pdf_filename: str | None = None,
) -> None:
    """
    Send an HTML email via Gmail SMTP using an App Password.
    Optionally attaches a PDF version of the same HTML content.
    """
    from_email = from_email or os.environ.get("GMAIL_ADDRESS")
    app_password = app_password or os.environ.get("GMAIL_APP_PASSWORD")

    if not from_email or not app_password:
        raise ValueError("Missing GMAIL_ADDRESS or GMAIL_APP_PASSWORD in env (or pass explicitly).")

    subject = (subject or "").replace("\xa0", " ").strip()
    html_body = (html_body or "").replace("\xa0", " ")
    
    msg = EmailMessage(policy=policy.SMTPUTF8)
    msg["Subject"] = str(Header(subject, "utf-8"))
    msg["From"] = from_email
    msg["To"] = to_email

    # plain-text fallback + HTML
    msg.set_content("This email contains an HTML report. Please view it in an HTML-capable email client.")
    msg.add_alternative(html_body, subtype="html")

    if attach_pdf:
        pdf_bytes = html_to_pdf_bytes(html_body)
        if not pdf_filename:
            pdf_filename = "weibo_report.pdf"
        pdf_filename = pdf_filename.replace("\xa0", " ")

        msg.add_attachment(
            pdf_bytes,
            maintype="application",
            subtype="pdf",
            filename=pdf_filename,
        )

    # Gmail SMTP
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(from_email, app_password)
        smtp.send_message(msg)

In [10]:
def monitor_weibo_post(
    COOKIE_HEADER: str,
    BASE: str,
    params: dict,
    start_time: datetime,
    end_time: datetime | None = None,
    *,
    max_pages: int = 100,
    rows_per_request_cap: int = 50,
    alert_callback=None,
    email_to: str | None = None,               # who receives the report
    email_enabled: bool = True,                # master switch
    email_subject_prefix: str = "[Weibo Monitoring]",      # subject prefix
) -> pd.DataFrame:
    """
    Monitoring loop with:
      - Fresh params per run (max_id reset)
      - Keyset change detection (handles delete+add)
      - DeepSeek scoring only for new comments
      - Alerts (visible only)
      - Scheduled reports AFTER the first run starting at/after milestone finishes
    """
    if end_time is None:
        end_time = start_time + timedelta(hours=96)

    report_times = build_report_schedule(start_time)
    report_idx = 0

    last_total_neg_alert = 0
    last_top5_neg_alert = 0

    prev_keyset = None

    seen_keys = set()
    df_scored_all = pd.DataFrame(
        columns=["posted_time", "comment", "comment_type", "sentiment", "confidence"]
    )

    last_visible_scored = None

    if alert_callback is None:
        def alert_callback(reason: str, report: dict):
            print("\n" + "!" * 90)
            print(f"[ALERT] {reason}")
            print("!" * 90)

    # ------------------ Local helper to email a report ------------------
    def _email_report_if_needed(report: dict):
        if not (email_enabled and email_to and report is not None):
            return
        try:
            html = render_report_dict_to_html(report)
            # Prefer report time if present; fallback to now
            rt = report.get("time", now_cn())
            if isinstance(rt, datetime):
                ts = rt.strftime("%Y-%m-%d %H:%M:%S")
            else:
                ts = str(rt)
            title = report.get("title", "Weibo Sentiment Report")
            subject = f"{email_subject_prefix} {title} | {ts}"
            send_gmail_html(
                subject=subject, 
                html_body=html, 
                to_email=email_to,
                attach_pdf=True,
                pdf_filename=build_dynamic_pdf_name(report)
            )
            print(f"[EMAIL] Sent report to: {email_to}")
        except Exception as e:
            print("\n" + "=" * 80)
            print(f"[EMAIL ERROR] Failed to send report to {email_to}")
            print("Exception message:", e)
            print("\nFull traceback:")
            traceback.print_exc()
            print("=" * 80 + "\n")

    # wait until start_time
    while now_cn() < start_time:
        time.sleep(1)

    while True:
        run_start = now_cn()

        # ---- fresh params each run (CRITICAL) ----
        params_run = params.copy()
        params_run["max_id"] = 0

        # ---- scrape ----
        try:
            comments = fetch_all(COOKIE_HEADER, BASE, params_run, max_pages=max_pages)
            df_latest = build_comments_df(comments)
        except Exception as e:
            print(f"[WARN] fetch_all/build_comments_df failed: {e}")
            df_latest = pd.DataFrame(columns=["posted_time", "comment", "comment_type"])

        # if empty scrape: still allow scheduled reports (using last snapshot), then sleep aligned
        if df_latest.empty:
            # scheduled reports AFTER run finishes, based on run_start
            while report_idx < len(report_times) and run_start >= report_times[report_idx]:
                rt = report_times[report_idx]
                if last_visible_scored is not None:
                    render_report_visible_only(
                        latest_visible_scored=last_visible_scored,
                        report_title=f"Scheduled report @ {rt.strftime('%Y-%m-%d %H:%M:%S')}"
                    )
                    _email_report_if_needed(report)
                else:
                    print(f"[REPORT] Scheduled report @ {rt.strftime('%Y-%m-%d %H:%M:%S')} (no data yet)")
                report_idx += 1

            next_poll_at = compute_next_poll_time(start_time, now_cn())
            sleep_s = (next_poll_at - now_cn()).total_seconds()
            if sleep_s > 0:
                time.sleep(sleep_s)
            continue

        # ---- keyset + unchanged detection ----
        df_latest = df_latest.copy()
        df_latest["_k"] = df_latest.apply(make_comment_key, axis=1)
        current_keyset = set(df_latest["_k"].tolist())
        unchanged = (prev_keyset is not None and current_keyset == prev_keyset)
        prev_keyset = current_keyset

        # build visible snapshot (cheap merge) for reporting
        latest_no_k = df_latest.drop(columns=["_k"], errors="ignore").copy()
        last_visible_scored = attach_sentiment_from_master(latest_no_k, df_scored_all)

        # ---- if changed: score new only ----
        if not unchanged:
            mask_new = ~df_latest["_k"].isin(seen_keys)
            new_rows_df = df_latest.loc[mask_new].drop(columns=["_k"]).copy()

            for k in df_latest.loc[mask_new, "_k"].tolist():
                seen_keys.add(k)

            if not new_rows_df.empty:
                try:
                    scored_new = score_new_comments_only(new_rows_df, rows_per_request_cap=rows_per_request_cap)
                    df_scored_all = pd.concat([df_scored_all, scored_new], ignore_index=True)
                except Exception as e:
                    print(f"[WARN] DeepSeek scoring failed: {e}")

            # refresh visible snapshot after scoring
            last_visible_scored = attach_sentiment_from_master(latest_no_k, df_scored_all)

            # ---- alerts (visible-only) after scoring ----
            visible_total_neg = int((last_visible_scored["sentiment"] == "NEG").sum()) if not last_visible_scored.empty else 0
            current_total_level = (visible_total_neg // 3) * 3

            if current_total_level < last_total_neg_alert:
                last_total_neg_alert = current_total_level

            if current_total_level >= 3 and current_total_level > last_total_neg_alert:
                last_total_neg_alert = current_total_level
                report = render_report_visible_only(
                    latest_visible_scored=last_visible_scored,
                    report_title=f"ALERT (Visible): Total NEG reached {last_total_neg_alert}"
                )
                _email_report_if_needed(report)
                alert_callback(f"(Visible) Total NEG reached {last_total_neg_alert}", report)

        # this part stays out of the unchanged loop because top comments can change based on like count
        top5_primary_visible = last_visible_scored[last_visible_scored["comment_type"] == "primary"].head(5).copy()
        visible_top5_neg = int((top5_primary_visible["sentiment"] == "NEG").sum()) if not top5_primary_visible.empty else 0

        if visible_top5_neg < last_top5_neg_alert:
            last_top5_neg_alert = visible_top5_neg

        if visible_top5_neg >= 1 and visible_top5_neg > last_top5_neg_alert:
            last_top5_neg_alert = visible_top5_neg
            report = render_report_visible_only(
                latest_visible_scored=last_visible_scored,
                report_title=f"ALERT (Visible): Top-5 primary NEG count = {last_top5_neg_alert}"
            )
            _email_report_if_needed(report)
            alert_callback(f"(Visible) Top-5 primary NEG count = {last_top5_neg_alert}", report)

        # ---- scheduled reports AFTER the run that starts at/after milestone finishes ----
        while report_idx < len(report_times) and run_start >= report_times[report_idx]:
            rt = report_times[report_idx]
            report = render_report_visible_only(
                latest_visible_scored=last_visible_scored,
                report_title=f"Scheduled report @ {rt.strftime('%Y-%m-%d %H:%M:%S')}"
            )
            _email_report_if_needed(report)
            report_idx += 1

        # ---- final boundary check ----
        if now_cn() >= end_time:
            print("Final report rendered. End of monitoring.")
            break

        # ---- sleep to next aligned boundary ----
        next_poll_at = compute_next_poll_time(start_time, now_cn())
        sleep_s = (next_poll_at - now_cn()).total_seconds()
        if sleep_s > 0:
            time.sleep(sleep_s)

    return df_scored_all

## 1. 获取微博的所有评论

In [11]:
CHROME_BINARY = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

# post_url = "https://weibo.com/1742666164/QsdsUmj1q"  # 唯一需要user input的地方
post_url = "https://weibo.com/1797932894/QuyO4kINa"
uid = post_url.rstrip("/").split("/")[-2]
driver, COOKIE_HEADER, status_id = open_weibo_and_get_cookie(post_url, timeout_seconds=120)

print("Cookie header length:", len(COOKIE_HEADER))
print("Extracted status_id:", status_id)
driver.quit()

Weibo login required.
A Chrome window has opened.
Please log in to Weibo (QR code / username).
You have 120 seconds (2 minutes).
After login, come back here — the script will continue automatically.
Cookie header length: 538
Extracted status_id: 5272836404938744


In [12]:
# Web scrap the post
BASE = "https://weibo.com/ajax/statuses/buildComments"

params = {
    "flow": 0, # 0按热度，1按时间排序
    "is_reload": 1,
    "id": f"{status_id}",
    "is_show_bulletin": 2,
    "is_mix": 0,
    "count": 20,
    "uid": f"{uid}",
    "fetch_level": 0,
    "max_id": 0,
}

# 这是我的个人API密钥，不可泄露
os.environ["DEEPSEEK_API_KEY"] = "sk-26e3bd6a00974081a25dc993e91f2a09"

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# email
os.environ["GMAIL_ADDRESS"] = "mclaneliu01@gmail.com"
os.environ["GMAIL_APP_PASSWORD"] = "vcihmmwzzbgngarh" # 这个也不可以泄露

In [13]:
# Example usage:
start_time = datetime(2026, 4, 22, 8, 54, 0, tzinfo=CN_TZ) #这里要放中国时间
df_all_history = monitor_weibo_post(
    COOKIE_HEADER=COOKIE_HEADER,
    BASE=BASE,
    params=params,
    start_time=start_time,
    max_pages=100,
    rows_per_request_cap=50,
    email_to="liuli@winpak.com.cn"
)

[primary] page=1 got=20 next_max_id=140264812329501
[primary] page=2 got=18 next_max_id=139302741850873
[primary] page=3 got=20 next_max_id=139302739405261
[primary] page=4 got=20 next_max_id=138890853054007
[primary] page=5 got=20 next_max_id=138890438752061
[primary] page=6 got=20 next_max_id=138615544583010
[primary] page=7 got=19 next_max_id=137516035254046
[primary] page=8 got=15 next_max_id=82540606944794
[primary] page=9 got=16 next_max_id=0
[replies] 22/168 primary comments have replies; fetching...
[replies] fetched 20/22 reply threads
[replies] done. Fetched 22 reply threads.


DeepSeek sentiment (3-class):   0%|          | 0/4 [00:00<?, ?it/s]


[REPORT] Scheduled report @ 2026-04-22 08:54:00
Time: 2026-04-22 08:57:42
------------------------------------------------------------------------------------------
Visible total=196 | POS=196 (100.0%)  NEG=0 (0.0%)  MIXED=0 (0.0%)  UNKNOWN=0 (0.0%)
所有评论=196 | 好评=196 (100.0%)  差评=0 (0.0%)  中性=0 (0.0%)  未评分=0 (0.0%)

Top-5 Comments by Popularity:
热评前5条：


,posted_time,sentiment,confidence,comment
0,2026-03-04 20:01:35,POS,0.95,代言人好美[太开心][太开心][太开心] http://t.cn/AXcFTzoE
2,2026-03-04 20:02:21,POS,0.9,和代言人文淇一起精简底妆[哇]#文淇阿玛尼美妆品牌代言人#
3,2026-03-04 20:01:43,POS,0.9,精简一点，也挺好看的！[哈哈]和#文淇阿玛尼美妆品牌代言人# 一起精简底妆[期待][期待]
5,2026-03-04 20:03:17,POS,0.9,#文淇阿玛尼美妆品牌代言人# 和文淇一起做自己！ http://t.cn/AXcFTskU
7,2026-03-04 20:03:23,POS,0.95,允许自己真实，比完美更有勇气。文淇×阿玛尼美妆，自信由我，自在发光#文淇阿玛尼美妆品牌代言人...



Duplicate Comment Summary:
重复评论总结：


,comment,freq,sentiment
0,和代言人文淇一起精简底妆[哇]#文淇阿玛尼美妆品牌代言人#,21,POS
1,@阿玛尼,19,POS
2,允许自己真实，比完美更有勇气。文淇×阿玛尼美妆，自信由我，自在发光#文淇阿玛尼美妆品牌代言人#,3,POS
3,和代言人文淇一起精简底妆[期待]#文淇阿玛尼美妆品牌代言人#,3,POS
4,文淇,3,POS



关键字总结:
关键词: 精简底妆
  - 和代言人文淇一起精简底妆[哇]#文淇阿玛尼美妆品牌代言人#
  - 精简一点，也挺好看的！[哈哈]和#文淇阿玛尼美妆品牌代言人# 一起精简底妆[期待][期待]
  - 和代言人@演员文淇 一起精简底妆 回归本身
关键词: 代言人文淇
  - 代言人好美[太开心][太开心][太开心] http://t.cn/AXcFTzoE
  - 代言人文淇和阿玛尼好适配！！
  - 代言人文淇状态超好
关键词: 真实自信
  - 允许自己真实，比完美更有勇气。文淇×阿玛尼美妆，自信由我，自在发光#文淇阿玛尼美妆品牌代言人#
  - 卸下伪装，忠于自我，以坦诚姿态，绽放原生生命力#文淇阿玛尼美妆品牌代言人##阿玛尼小白娇妆前乳# http://t.cn/AXcFHMOb
  - #文淇阿玛尼美妆品牌代言人#真实自有万钧之力，精致只为悦己。文淇与阿玛尼小白娇，诠释自在本真之美 http://t.cn/AXcFTrew

总结: 评论主要围绕代言人文淇的美貌与阿玛尼品牌的契合度展开，强调精简底妆的理念和真实自信的品牌价值观。
[EMAIL] Sent report to: liuli@winpak.com.cn


KeyboardInterrupt: 